# CE-HyperKG-Ret Proposed V6 Model

 Uses MedImageInsight, 512 x 512 input, train-only CUI vocabulary with min frequency 5, top-N visual candidates 200, top-L predicted CUIs 10, CUI threshold 0.30, seeds 42/123/2025, and the Table 2 uncertainty-aware fusion rule.

In [ ]:
import os, re, gc, ast, sys, json, math, time, base64, random, shutil, warnings, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Tuple, Set, Optional

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, ImageOps, ImageDraw
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

try:
    from scipy.sparse import csr_matrix
except Exception as e:
    raise ImportError("scipy is required for efficient metric computation. Install scipy.") from e

try:
    from sklearn.manifold import TSNE
    from sklearn.decomposition import PCA
except Exception:
    TSNE = None
    PCA = None

try:
    from IPython.display import display
except Exception:
    display = lambda x: print(x.to_string() if hasattr(x, "to_string") else x)


@dataclass
class CFG:

    DATA_ROOT: str = os.environ.get("DATA_ROOT", "../data/roco_v2")
    OUT_DIR: str = os.environ.get("OUTPUT_DIR", os.path.join(os.environ.get("OUTPUT_ROOT", "../results"), "ce_hyperkg_ret_v6"))


    HF_REPO_ID: str = "lion-ai/MedImageInsights"
    MODEL_CACHE_DIR: str = os.environ.get("MODEL_CACHE_DIR", "../models/medimageinsight_model_cache")
    MODEL_TAG: str = "medimageinsight"
    INSTALL_MEDIMAGEINSIGHT: bool = True
    REUSE_CACHED_FEATURES: bool = True


    TRAIN_MEDIMAGEINSIGHT_ENCODER: bool = True
    FINETUNE_MODE: str = "all"
    UNFREEZE_LAST_N_CHILDREN: int = 3
    IMAGE_SIZE: int = 512
    ENCODER_LR: float = 1e-4
    HEAD_LR: float = 1e-4


    MIN_CUI_FREQ: int = 5
    MAX_CUIS: int = 1916
    MAX_TRAIN_SAMPLES: Optional[int] = None
    MAX_VALID_SAMPLES: Optional[int] = None
    MAX_TEST_SAMPLES: Optional[int] = None


    EMBED_BATCH_SIZE: int = 8
    NUM_WORKERS: int = 0


    SEEDS: Tuple[int, ...] = (42, 123, 2025)
    EPOCHS: int = 25
    BATCH_SIZE: int = 32
    WEIGHT_DECAY: float = 1e-4
    HEAD_DROPOUT: float = 0.20
    PROJECTION_DIM: int = 512
    USE_AMP: bool = True
    GRAD_CLIP: float = 1.0
    EARLY_STOP_PATIENCE: int = 5
    VALIDATION_METRIC: str = "generic_filtered_ndcg_at_5"


    LAMBDA_BCE: float = 1.0
    LAMBDA_RETRIEVAL: float = 1.0
    LAMBDA_COUNTERFACTUAL: float = 0.10
    LAMBDA_KG_RANK: float = 0.10
    KG_RANK_MARGIN: float = 0.20
    SUPCON_TEMP: float = 0.07
    COUNTERFACTUAL_MARGIN: float = 0.20
    POS_WEIGHT_MAX: float = 20.0


    QUERY_SPLIT: str = "test"
    DATABASE_SPLIT: str = "test"
    SAME_SPLIT_SELF_EXCLUDE: bool = True
    TOP_N_VISUAL: int = 200
    TOP_EXPLAIN: int = 10
    K_VALUES: Tuple[int, ...] = (5, 10, 50)
    K_CURVE_MAX: int = 50


    PRED_CUI_THRESHOLD: float = 0.30
    PRED_TOP_M: int = 10
    KG_EXPAND_HOPS: int = 1
    KG_MAX_NEIGHBORS_PER_CUI: int = 30
    KG_EXPAND_DECAY: float = 0.60


    

    BOOTSTRAP_ROUNDS: int = 10000
    RANDOM_EXAMPLE_QUERIES: int = 6
    FAILURE_EXAMPLE_QUERIES: int = 6
    TSNE_MAX_POINTS: int = 1500


    SAVE_ZIP: bool = True
    FUSION_RULE: Tuple[Tuple[float, float, float, float], ...] = ((0.40, 0.70, 0.20, 0.10), (0.70, 0.50, 0.30, 0.20), (1.01, 0.30, 0.35, 0.35))


cfg = CFG()


def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def save_json(obj: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def savefig(fig_dir: str, name: str) -> None:
    path = os.path.join(fig_dir, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", path)


def pip_install(args: List[str]) -> None:
    print("Installing:", " ".join(args))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)


def cosine_normalize(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)


def parse_cuis_from_value(x: Any) -> List[str]:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, (list, tuple, set)):
        raw = " ".join(map(str, x))
    else:
        raw = str(x)
        if raw.strip().startswith(("[", "(")):
            try:
                val = ast.literal_eval(raw)
                if isinstance(val, (list, tuple, set)):
                    raw = " ".join(map(str, val))
            except Exception:
                pass
    cuis = re.findall(r"C\d{7,8}", raw.upper())
    seen, out = set(), []
    for c in cuis:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out


def jaccard_sets(a: Set[int], b: Set[int]) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / max(len(a | b), 1)


def dcg_at_k(gains: List[float], k: int) -> float:
    gains = np.asarray(gains[:k], dtype=np.float64)
    if len(gains) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(gains) + 2))
    return float(np.sum(gains / discounts))


def ndcg_at_k(predicted_gains: List[float], ideal_gains_sorted: np.ndarray, k: int) -> float:
    ideal = dcg_at_k(ideal_gains_sorted, k)
    if ideal <= 0:
        return np.nan
    return dcg_at_k(predicted_gains, k) / ideal


def average_precision_at_k(binary_rel: List[bool], total_rel: int, k: int) -> float:
    if total_rel <= 0:
        return np.nan
    hits = 0
    precisions = []
    for rank, is_rel in enumerate(binary_rel[:k], start=1):
        if is_rel:
            hits += 1
            precisions.append(hits / rank)
    denom = min(total_rel, k)
    return float(np.sum(precisions) / max(denom, 1)) if precisions else 0.0


def reciprocal_rank_at_k(binary_rel: List[bool], k: int) -> float:
    for rank, is_rel in enumerate(binary_rel[:k], start=1):
        if is_rel:
            return 1.0 / rank
    return 0.0


IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def find_data_root(preferred: str) -> str:
    if Path(preferred).exists():
        return preferred
    print("DATA_ROOT not found. Searching DATA_SEARCH_DIR for train/valid/test_concepts.csv ...")
    candidates = []
    for root, dirs, files in os.walk(os.environ.get("DATA_SEARCH_DIR", "../data")):
        if {"train_concepts.csv", "valid_concepts.csv", "test_concepts.csv"}.issubset(set(files)):
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            "Could not find a folder containing train_concepts.csv, valid_concepts.csv, and test_concepts.csv. "
            "Set cfg.DATA_ROOT correctly."
        )
    candidates = sorted(candidates, key=len)
    return candidates[0]


def read_csv_flexible(path: str) -> pd.DataFrame:
    for sep in [",", "\t", ";", "|"]:
        try:
            df = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False)
            if df.shape[1] >= 1:
                return df
        except Exception:
            pass
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def build_image_map(data_root: str, split: str) -> Dict[str, str]:
    split_dir = Path(data_root) / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Missing image folder: {split_dir}")
    mp = {}
    for p in split_dir.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            mp[p.stem] = str(p)
            mp[p.name] = str(p)
            mp[p.stem.lower()] = str(p)
            mp[p.name.lower()] = str(p)
    return mp


def choose_id_column(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    preferred = [
        "image_id", "id", "sample_id", "image", "filename", "file_name", "name",
        "image_name", "ImageID", "ID"
    ]
    for c in preferred:
        if c in cols:
            return c
    for c in cols:
        lc = c.lower()
        if "image" in lc or "file" in lc or lc.endswith("id"):
            return c
    return cols[0]


def load_one_split(data_root: str, split: str, max_samples: Optional[int] = None) -> pd.DataFrame:
    csv_path = Path(data_root) / f"{split}_concepts.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing: {csv_path}")
    raw = read_csv_flexible(str(csv_path))
    image_map = build_image_map(data_root, split)
    id_col = choose_id_column(raw)

    rows = []
    missing_image = 0
    missing_cui = 0
    for _, r in raw.iterrows():
        raw_id = str(r.get(id_col, "")).strip()
        candidates = [raw_id, Path(raw_id).stem, Path(raw_id).name, raw_id.lower(), Path(raw_id).stem.lower(), Path(raw_id).name.lower()]
        image_path = None
        for key in candidates:
            if key in image_map:
                image_path = image_map[key]
                break
        if image_path is None:
            missing_image += 1
            continue


        all_text = " ".join(str(v) for v in r.values.tolist())
        cuis = parse_cuis_from_value(all_text)
        if len(cuis) == 0:
            missing_cui += 1
            continue

        rows.append({
            "sample_id": Path(image_path).stem,
            "image_id": Path(image_path).stem,
            "image_path": image_path,
            "cui_list": cuis,
            "split": split,
        })

    df = pd.DataFrame(rows)
    if max_samples is not None and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42).reset_index(drop=True)
    print(f"{split}: raw={len(raw)} usable={len(df)} missing_image={missing_image} missing_cui={missing_cui}")
    if len(df) == 0:
        print("Columns were:", raw.columns.tolist())
        print(raw.head())
        raise ValueError(f"No usable rows for split={split}. Check image IDs and CUI columns.")
    return df.reset_index(drop=True)


def load_roco_splits(cfg: CFG) -> Dict[str, pd.DataFrame]:
    data_root = find_data_root(cfg.DATA_ROOT)
    cfg.DATA_ROOT = data_root
    print("Using DATA_ROOT:", data_root)
    splits = {
        "train": load_one_split(data_root, "train", cfg.MAX_TRAIN_SAMPLES),
        "valid": load_one_split(data_root, "valid", cfg.MAX_VALID_SAMPLES),
        "test": load_one_split(data_root, "test", cfg.MAX_TEST_SAMPLES),
    }
    print("Example row:", splits["train"].head(1).to_dict("records"))
    return splits


def build_cui_vocab(train_df: pd.DataFrame, cfg: CFG) -> Tuple[Dict[str, int], List[str]]:
    counter = Counter()
    for cuis in train_df["cui_list"]:
        counter.update(cuis)
    items = [(c, n) for c, n in counter.most_common() if n >= cfg.MIN_CUI_FREQ]
    if cfg.MAX_CUIS is not None:
        items = items[:cfg.MAX_CUIS]
    if len(items) == 0:
        raise ValueError("CUI vocabulary became empty. Set cfg.MIN_CUI_FREQ=1.")
    inv_vocab = [c for c, _ in items]
    vocab = {c: i for i, c in enumerate(inv_vocab)}
    print(f"CUI vocab size from train only: {len(vocab)}")
    return vocab, inv_vocab


def add_label_indices(df: pd.DataFrame, vocab: Dict[str, int]) -> pd.DataFrame:
    out = df.copy()
    out["label_indices"] = out["cui_list"].apply(lambda xs: sorted({vocab[c] for c in xs if c in vocab}))
    before = len(out)
    out = out[out["label_indices"].apply(len) > 0].reset_index(drop=True)
    print(f"Kept {len(out)}/{before} rows with at least one train-vocab CUI for split={out['split'].iloc[0] if len(out) else '?'}")
    return out


def ensure_medimageinsight_package(cfg: CFG):
    try:
        from MedImageInsights.Models.medimageinsightmodel import MedImageInsight
        return MedImageInsight
    except Exception as e:
        print("MedImageInsights import failed before install:", repr(e))

    if not cfg.INSTALL_MEDIMAGEINSIGHT:
        raise ImportError("MedImageInsights is not installed and cfg.INSTALL_MEDIMAGEINSIGHT=False")


    pip_install(["MedImageInsights==0.1.1", "--no-deps"])
    for dep in ["huggingface_hub", "hf_xet", "einops", "fvcore", "yacs", "timm", "transformers", "safetensors", "tenacity", "mup"]:
        try:
            pip_install([dep])
        except Exception as dep_e:
            print("Could not install", dep, "->", repr(dep_e))

    from MedImageInsights.Models.medimageinsightmodel import MedImageInsight
    return MedImageInsight


MODEL_DIR_NAME_CANDIDATES = ["v20240927", "2024.09.27", "2024_09_27"]
VISION_WEIGHT_CANDIDATES = ["medimageinsight-v1.0.0.pt", "medimageinsigt-v1.0.0.pt"]


def find_candidate_model_dirs(base: str) -> List[Path]:
    base = Path(base)
    candidates = []
    if not base.exists():
        return candidates
    if (base / "config.yaml").exists():
        candidates.append(base)
    for p in base.rglob("config.yaml"):
        candidates.append(p.parent)
    candidates = sorted(set(candidates), key=lambda x: (x.name not in MODEL_DIR_NAME_CANDIDATES, len(str(x))))
    return candidates


def locate_medimageinsight_model_dir(cfg: CFG) -> Tuple[str, str]:
    from huggingface_hub import snapshot_download

    roots_to_check = []
    env_dir = os.environ.get("MEDIMAGEINSIGHT_MODEL_DIR")
    if env_dir:
        roots_to_check.append(env_dir)
    try:
        import MedImageInsights
        pkg_root = Path(MedImageInsights.__file__).resolve().parent
        roots_to_check += [str(pkg_root), str(pkg_root.parent)]
    except Exception:
        pass
    roots_to_check += [cfg.MODEL_CACHE_DIR, os.environ.get("DATA_SEARCH_DIR", "../data"), os.environ.get("OUTPUT_ROOT", "../results")]

    for root in roots_to_check:
        for d in find_candidate_model_dirs(root):
            for weight_name in VISION_WEIGHT_CANDIDATES:
                if (d / "vision_model" / weight_name).exists():
                    return str(d), weight_name

    print("Downloading MedImageInsight model files from Hugging Face. Turn Kaggle Internet ON for the first run.")
    local_dir = os.path.join(cfg.MODEL_CACHE_DIR, "lion-ai-MedImageInsights")
    snapshot_download(
        repo_id=cfg.HF_REPO_ID,
        local_dir=local_dir,
        local_dir_use_symlinks=False,
        allow_patterns=[
            "2024.09.27/config.yaml",
            "2024.09.27/language_model/clip_tokenizer_4.16.2/**",
            "2024.09.27/vision_model/**",
            "*.md",
        ],
    )
    for d in find_candidate_model_dirs(local_dir):
        for weight_name in VISION_WEIGHT_CANDIDATES:
            if (d / "vision_model" / weight_name).exists():
                return str(d), weight_name
    dirs = find_candidate_model_dirs(local_dir)
    if dirs:
        return str(dirs[0]), VISION_WEIGHT_CANDIDATES[0]
    raise FileNotFoundError("Could not locate MedImageInsight model files.")


def safe_link_or_copy(src: Path, dst: Path) -> None:
    src, dst = Path(src), Path(dst)
    if dst.exists() or dst.is_symlink():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(str(src), str(dst), target_is_directory=src.is_dir())
    except Exception:
        if src.is_dir():
            shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
        else:
            shutil.copy2(str(src), str(dst))


def prepare_medimageinsight_compatible_dir(model_dir: str, cfg: CFG) -> str:
    from huggingface_hub import snapshot_download

    src_dir = Path(model_dir).resolve()
    official_tok = src_dir / "language_model" / "clip_tokenizer_4.16.2"
    wrapper_tok = src_dir / "language_model" / "clip_tokenizer_v4162"

    if wrapper_tok.exists():
        return str(src_dir)

    if official_tok.exists():
        try:
            safe_link_or_copy(official_tok, wrapper_tok)
            if wrapper_tok.exists():
                print("Created tokenizer alias:", wrapper_tok)
                return str(src_dir)
        except Exception as e:
            print("Could not write tokenizer alias; creating compatibility folder:", repr(e))

    if not official_tok.exists():
        print("Tokenizer folder missing. Downloading tokenizer/config/vision files from Hugging Face...")
        dl_root = Path(cfg.MODEL_CACHE_DIR) / "lion-ai-MedImageInsights"
        snapshot_download(
            repo_id=cfg.HF_REPO_ID,
            local_dir=str(dl_root),
            local_dir_use_symlinks=False,
            allow_patterns=[
                "2024.09.27/config.yaml",
                "2024.09.27/language_model/clip_tokenizer_4.16.2/**",
                "2024.09.27/vision_model/**",
            ],
        )
        downloaded_model_dir = dl_root / "2024.09.27"
        downloaded_tok = downloaded_model_dir / "language_model" / "clip_tokenizer_4.16.2"
        if downloaded_tok.exists():
            official_tok = downloaded_tok
            if (downloaded_model_dir / "vision_model").exists():
                src_dir = downloaded_model_dir
        else:
            raise FileNotFoundError("Could not find MedImageInsight tokenizer folder after download.")

    compat_dir = Path(cfg.MODEL_CACHE_DIR) / "compatible_2024.09.27"
    compat_dir.mkdir(parents=True, exist_ok=True)
    if not (src_dir / "config.yaml").exists():
        raise FileNotFoundError(f"Missing config.yaml in {src_dir}")
    if not (src_dir / "vision_model").exists():
        raise FileNotFoundError(f"Missing vision_model folder in {src_dir}")
    safe_link_or_copy(src_dir / "config.yaml", compat_dir / "config.yaml")
    safe_link_or_copy(src_dir / "vision_model", compat_dir / "vision_model")
    (compat_dir / "language_model").mkdir(parents=True, exist_ok=True)
    safe_link_or_copy(official_tok, compat_dir / "language_model" / "clip_tokenizer_4.16.2")
    safe_link_or_copy(official_tok, compat_dir / "language_model" / "clip_tokenizer_v4162")
    print("Using MedImageInsight compatibility model_dir:", compat_dir)
    return str(compat_dir)


def load_medimageinsight(cfg: CFG, device: torch.device):
    MedImageInsight = ensure_medimageinsight_package(cfg)
    try:
        from huggingface_hub import snapshot_download
    except Exception:
        pip_install(["huggingface_hub", "hf_xet"])

    model_dir, vision_model_name = locate_medimageinsight_model_dir(cfg)
    model_dir = prepare_medimageinsight_compatible_dir(model_dir, cfg)
    print("MedImageInsight model_dir:", model_dir)
    print("MedImageInsight vision_model_name:", vision_model_name)

    classifier = MedImageInsight(
        model_dir=model_dir,
        vision_model_name=vision_model_name,
        language_model_name="language_model.pth",
    )
    classifier.load_model()
    classifier.model.eval()
    for p in classifier.model.parameters():
        p.requires_grad = False
    return classifier


class RocoPathDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.paths = df["image_path"].astype(str).tolist()
        self.ids = df["image_id"].astype(str).tolist()
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        return self.paths[idx], self.ids[idx]


def collate_paths(batch):
    paths, ids = zip(*batch)
    return list(paths), list(ids)


def image_file_to_base64(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


@torch.no_grad()
def extract_medimageinsight_features(classifier, df: pd.DataFrame, split_name: str, cfg: CFG, feature_dir: str) -> Tuple[np.ndarray, pd.DataFrame]:
    feat_path = os.path.join(feature_dir, f"{split_name}_{cfg.MODEL_TAG}_features.npy")
    id_path = os.path.join(feature_dir, f"{split_name}_{cfg.MODEL_TAG}_ids.csv")

    if cfg.REUSE_CACHED_FEATURES and os.path.exists(feat_path) and os.path.exists(id_path):
        feats = np.load(feat_path).astype("float32")
        ids = pd.read_csv(id_path)["image_id"].astype(str).tolist()
        if len(ids) == len(feats):
            id_to_row = {str(r.image_id): i for i, r in df.reset_index(drop=True).iterrows()}
            if all(i in id_to_row for i in ids):
                aligned_df = pd.DataFrame([df.iloc[id_to_row[i]] for i in ids]).reset_index(drop=True)
                print(f"Loaded cached {split_name} MedImageInsight features:", feats.shape)
                return feats, aligned_df
            print(f"Cache ids do not match current {split_name} df; recomputing.")

    loader = DataLoader(
        RocoPathDataset(df),
        batch_size=cfg.EMBED_BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=False,
        collate_fn=collate_paths,
    )

    feats, ids = [], []
    classifier.model.eval()
    for step, (paths, image_ids) in enumerate(tqdm(loader, desc=f"Encoding {split_name}"), start=1):
        images_b64, good_ids = [], []
        for p, img_id in zip(paths, image_ids):
            try:
                images_b64.append(image_file_to_base64(p))
                good_ids.append(img_id)
            except Exception as e:
                print("Skipping unreadable image:", p, repr(e))
        if not images_b64:
            continue
        out = classifier.encode(images=images_b64)
        emb = np.asarray(out["image_embeddings"], dtype="float32")
        if emb.ndim == 1:
            emb = emb[None, :]
        feats.append(emb)
        ids.extend(good_ids)
        if step % 25 == 0:
            print(f"{split_name}: encoded {len(ids)}/{len(df)}")

    if not feats:
        raise RuntimeError(f"No features extracted for split={split_name}")
    feats = np.concatenate(feats, axis=0).astype("float32")
    np.save(feat_path, feats)
    pd.DataFrame({"image_id": ids}).to_csv(id_path, index=False)

    id_to_row = {str(r.image_id): i for i, r in df.reset_index(drop=True).iterrows()}
    aligned_df = pd.DataFrame([df.iloc[id_to_row[i]] for i in ids if i in id_to_row]).reset_index(drop=True)
    print(f"Saved {split_name} MedImageInsight features:", feats.shape)
    return feats, aligned_df


class FeatureCUIDataset(Dataset):
    def __init__(self, feats: np.ndarray, df: pd.DataFrame, n_labels: int):
        self.feats = feats.astype("float32")
        self.label_indices = df["label_indices"].tolist()
        self.n_labels = n_labels
    def __len__(self):
        return len(self.feats)
    def __getitem__(self, idx):
        y = torch.zeros(self.n_labels, dtype=torch.float32)
        y[self.label_indices[idx]] = 1.0
        return torch.from_numpy(self.feats[idx]), y


class CUIProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int, n_labels: int, dropout: float = 0.2):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.projector = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, proj_dim),
        )
        self.classifier = nn.Linear(proj_dim, n_labels)
    def forward(self, x, return_embedding: bool = False):
        z = self.projector(self.norm(x))
        z = F.normalize(z, dim=1)
        if return_embedding:
            return z
        logits = self.classifier(z)
        return z, logits


def supervised_contrastive_multilabel_loss(emb: torch.Tensor, labels: torch.Tensor, temp: float = 0.07) -> torch.Tensor:

    b = emb.size(0)
    if b <= 1:
        return emb.float().new_tensor(0.0)
    emb_f = F.normalize(emb.float(), dim=1)
    labels_f = labels.float()
    sim = emb_f @ emb_f.T / max(float(temp), 1e-6)
    eye = torch.eye(b, device=emb.device, dtype=torch.bool)
    overlap = (labels_f @ labels_f.T) > 0
    pos_mask = overlap & (~eye)
    if pos_mask.sum() == 0:
        return emb_f.new_tensor(0.0)
    sim = sim.masked_fill(eye, -1e4)
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos_float = pos_mask.float()
    loss_i = -(log_prob * pos_float).sum(dim=1) / (pos_float.sum(dim=1) + 1e-12)
    valid = pos_mask.sum(dim=1) > 0
    return loss_i[valid].mean()


def counterfactual_margin_loss(emb: torch.Tensor, labels: torch.Tensor, margin: float = 0.2) -> torch.Tensor:

    b = emb.size(0)
    if b <= 2:
        return emb.float().new_tensor(0.0)
    emb_f = F.normalize(emb.float(), dim=1)
    labels_f = labels.float()
    sim = emb_f @ emb_f.T
    eye = torch.eye(b, device=emb.device, dtype=torch.bool)
    overlap = (labels_f @ labels_f.T) > 0
    pos_mask = overlap & (~eye)
    neg_mask = (~overlap) & (~eye)
    losses = []
    for i in range(b):
        if pos_mask[i].sum() == 0 or neg_mask[i].sum() == 0:
            continue
        pos_sim = sim[i][pos_mask[i]].min()
        neg_sim = sim[i][neg_mask[i]].max()
        losses.append(F.relu(float(margin) + neg_sim - pos_sim))
    if not losses:
        return emb_f.new_tensor(0.0)
    return torch.stack(losses).mean()


def build_pos_weight(train_df: pd.DataFrame, n_labels: int, cfg: CFG, device: torch.device) -> torch.Tensor:
    pos_counts = np.zeros(n_labels, dtype=np.float32)
    for inds in train_df["label_indices"]:
        pos_counts[inds] += 1
    pos_weight = (len(train_df) - pos_counts) / np.maximum(pos_counts, 1)
    pos_weight = np.clip(pos_weight, 1.0, cfg.POS_WEIGHT_MAX)
    return torch.tensor(pos_weight, dtype=torch.float32, device=device)


def train_one_seed(
    seed: int,
    feats: Dict[str, np.ndarray],
    dfs: Dict[str, pd.DataFrame],
    n_labels: int,
    cfg: CFG,
    out_dir: str,
    fig_dir: str,
    device: torch.device,
) -> Tuple[CUIProjectionHead, pd.DataFrame]:
    seed_everything(seed)
    in_dim = int(feats["train"].shape[1])
    model = CUIProjectionHead(in_dim, cfg.PROJECTION_DIM, n_labels, cfg.HEAD_DROPOUT).to(device)
    pos_weight = build_pos_weight(dfs["train"], n_labels, cfg, device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.HEAD_LR, weight_decay=cfg.WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.USE_AMP and device.type == "cuda"))

    train_loader = DataLoader(
        FeatureCUIDataset(feats["train"], dfs["train"], n_labels),
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )
    valid_loader = DataLoader(
        FeatureCUIDataset(feats["valid"], dfs["valid"], n_labels),
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    best_valid = float("inf")
    best_path = os.path.join(out_dir, f"best_head_seed_{seed}.pt")
    patience = 0
    history = []

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        train_losses, train_bce, train_sup, train_cf = [], [], [], []
        for x, y in tqdm(train_loader, desc=f"seed {seed} epoch {epoch} train"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(cfg.USE_AMP and device.type == "cuda")):
                emb, logits = model(x)
                loss_bce = criterion(logits, y)
                loss_sup = supervised_contrastive_multilabel_loss(emb, y, cfg.SUPCON_TEMP)
                loss_cf = counterfactual_margin_loss(emb, y, cfg.COUNTERFACTUAL_MARGIN)
                loss = cfg.LAMBDA_BCE * loss_bce + cfg.LAMBDA_RETRIEVAL * loss_sup + cfg.LAMBDA_COUNTERFACTUAL * loss_cf
            scaler.scale(loss).backward()
            if cfg.GRAD_CLIP is not None and cfg.GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.detach().cpu()))
            train_bce.append(float(loss_bce.detach().cpu()))
            train_sup.append(float(loss_sup.detach().cpu()))
            train_cf.append(float(loss_cf.detach().cpu()))

        model.eval()
        valid_losses = []
        with torch.no_grad():
            for x, y in tqdm(valid_loader, desc=f"seed {seed} epoch {epoch} valid"):
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                emb, logits = model(x)
                loss_bce = criterion(logits, y)
                valid_losses.append(float(loss_bce.detach().cpu()))
        row = {
            "seed": seed,
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "train_bce": float(np.mean(train_bce)),
            "train_supcon": float(np.mean(train_sup)),
            "train_counterfactual": float(np.mean(train_cf)),
            "valid_bce": float(np.mean(valid_losses)),
        }
        history.append(row)
        print(row)

        if row["valid_bce"] < best_valid:
            best_valid = row["valid_bce"]
            torch.save(model.state_dict(), best_path)
            patience = 0
        else:
            patience += 1
            if patience >= cfg.EARLY_STOP_PATIENCE:
                print("Early stopping.")
                break

    model.load_state_dict(torch.load(best_path, map_location=device))
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(out_dir, f"training_history_seed_{seed}.csv"), index=False)

    plt.figure(figsize=(7, 4))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train total")
    plt.plot(hist_df["epoch"], hist_df["valid_bce"], marker="o", label="valid BCE")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training/validation loss curve seed {seed}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    savefig(fig_dir, f"01_training_validation_loss_seed_{seed}.png")
    return model, hist_df


@torch.no_grad()
def infer_head(model: CUIProjectionHead, feats: np.ndarray, batch_size: int, device: torch.device) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    embs, logits_all = [], []
    for s in tqdm(range(0, len(feats), batch_size), desc="Head inference"):
        x = torch.from_numpy(feats[s:s+batch_size].astype("float32")).to(device)
        z, logits = model(x)
        embs.append(z.detach().cpu().numpy().astype("float32"))
        logits_all.append(logits.detach().cpu().numpy().astype("float32"))
    embs = np.concatenate(embs, axis=0)
    logits_all = np.concatenate(logits_all, axis=0)
    probs = 1.0 / (1.0 + np.exp(-logits_all))
    return embs, probs.astype("float32"), logits_all


def build_train_hypergraph(train_df: pd.DataFrame, n_labels: int, cfg: CFG):
    hyperedges = []
    pair_counter = Counter()
    cui_freq = Counter()
    for inds in train_df["label_indices"]:
        nodes = sorted(set(int(i) for i in inds))
        if not nodes:
            continue
        hyperedges.append(nodes)
        cui_freq.update(nodes)
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                a, b = nodes[i], nodes[j]
                pair_counter[(a, b)] += 1
                pair_counter[(b, a)] += 1

    adjacency = defaultdict(list)
    for (a, b), w in pair_counter.items():
        adjacency[a].append((b, float(w)))
    for a in list(adjacency.keys()):
        adjacency[a] = sorted(adjacency[a], key=lambda x: x[1], reverse=True)[:cfg.KG_MAX_NEIGHBORS_PER_CUI]

    print(f"Built train-only hypergraph: {len(hyperedges)} hyperedges, {len(cui_freq)} CUI nodes.")
    return {"hyperedges": hyperedges, "adjacency": dict(adjacency), "cui_freq": dict(cui_freq)}


def top_predicted_cuis(prob: np.ndarray, cfg: CFG) -> List[Tuple[int, float]]:
    if len(prob) == 0:
        return []
    order = np.argsort(-prob)[:cfg.PRED_TOP_M]
    selected = [(int(i), float(prob[i])) for i in order if float(prob[i]) >= cfg.PRED_CUI_THRESHOLD]
    return selected


def expand_cuis(preds: List[Tuple[int, float]], graph: Dict[str, Any], cfg: CFG) -> Dict[int, float]:
    weights = {int(i): float(w) for i, w in preds}
    frontier = dict(weights)
    adjacency = graph["adjacency"]
    for _ in range(cfg.KG_EXPAND_HOPS):
        new_frontier = {}
        for cui, base_w in frontier.items():
            neigh = adjacency.get(int(cui), [])
            if not neigh:
                continue
            max_w = max(w for _, w in neigh) if neigh else 1.0
            for nb, edge_w in neigh:
                add = base_w * cfg.KG_EXPAND_DECAY * (edge_w / max(max_w, 1e-12))
                if add > weights.get(int(nb), 0.0):
                    weights[int(nb)] = float(add)
                    new_frontier[int(nb)] = float(add)
        frontier = new_frontier
        if not frontier:
            break
    s = sum(weights.values()) + 1e-12
    return {k: float(v / s) for k, v in weights.items()}


def confidence_from_preds(preds: List[Tuple[int, float]]) -> float:
    if not preds:
        return 0.0
    vals = np.array([p for _, p in preds], dtype=np.float32)
    return float(np.clip(vals.max(), 0.0, 1.0))


def make_sparse_label_matrix(df: pd.DataFrame, n_labels: int) -> csr_matrix:
    rows, cols = [], []
    for i, inds in enumerate(df["label_indices"]):
        for j in inds:
            rows.append(i)
            cols.append(int(j))
    data = np.ones(len(rows), dtype=np.int8)
    return csr_matrix((data, (rows, cols)), shape=(len(df), n_labels), dtype=np.int8)


def retrieve_and_explain(
    query_df: pd.DataFrame,
    db_df: pd.DataFrame,
    query_emb: np.ndarray,
    db_emb: np.ndarray,
    query_probs: np.ndarray,
    db_probs: np.ndarray,
    graph: Dict[str, Any],
    inv_vocab: List[str],
    cfg: CFG,
) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    qE = cosine_normalize(query_emb.astype("float32"))
    dE = cosine_normalize(db_emb.astype("float32"))
    qP = query_probs.astype("float32")
    dP = db_probs.astype("float32")
    qP_norm = cosine_normalize(qP + 1e-8)
    dP_norm = cosine_normalize(dP + 1e-8)

    same_pool = cfg.SAME_SPLIT_SELF_EXCLUDE and list(query_df["image_id"]) == list(db_df["image_id"])
    max_k = max(max(cfg.K_VALUES), cfg.TOP_EXPLAIN)
    all_indices = np.zeros((len(query_df), max_k), dtype=np.int64)
    all_scores = np.zeros((len(query_df), max_k), dtype=np.float32)
    explanation_rows = []

    for qi in tqdm(range(len(query_df)), desc="Retrieval + KG rerank"):
        visual = qE[qi] @ dE.T
        if same_pool:
            visual[qi] = -1e9
        top_n = min(cfg.TOP_N_VISUAL, len(db_df))
        cand = np.argpartition(-visual, kth=min(top_n - 1, len(visual) - 1))[:top_n]
        cand = cand[np.argsort(-visual[cand])]

        preds = top_predicted_cuis(qP[qi], cfg)
        expanded = expand_cuis(preds, graph, cfg)
        conf = confidence_from_preds(preds)


        if conf < 0.40:
            alpha, beta, gamma = 0.70, 0.20, 0.10
        elif conf < 0.70:
            alpha, beta, gamma = 0.50, 0.30, 0.20
        else:
            alpha, beta, gamma = 0.30, 0.35, 0.35

        v = visual[cand].astype("float32")
        v_norm = (v - v.min()) / (v.max() - v.min() + 1e-12)
        pred_sim = qP_norm[qi] @ dP_norm[cand].T
        if expanded:
            keys = np.array(list(expanded.keys()), dtype=np.int64)
            vals = np.array([expanded[k] for k in keys], dtype=np.float32)
            kg_score = dP[cand][:, keys] @ vals
        else:
            kg_score = np.zeros(len(cand), dtype=np.float32)

        final = alpha * v_norm + beta * pred_sim + gamma * kg_score
        order = np.argsort(-final)
        ranked = cand[order]
        ranked_scores = final[order]

        all_indices[qi, :] = ranked[:max_k]
        all_scores[qi, :] = ranked_scores[:max_k]

        q_gt = set(query_df.iloc[qi]["label_indices"])
        q_pred_names = [inv_vocab[i] for i, _ in preds[:cfg.TOP_EXPLAIN] if i < len(inv_vocab)]
        expanded_names = [inv_vocab[i] for i, _ in sorted(expanded.items(), key=lambda x: -x[1])[:cfg.TOP_EXPLAIN] if i < len(inv_vocab)]

        for local_rank, pi in enumerate(ranked[:cfg.TOP_EXPLAIN], start=1):
            c_gt = set(db_df.iloc[int(pi)]["label_indices"])
            shared = sorted(q_gt & c_gt)
            cand_preds = top_predicted_cuis(dP[int(pi)], cfg)[:cfg.TOP_EXPLAIN]
            explanation_rows.append({
                "query_index": qi,
                "query_image_id": query_df.iloc[qi]["image_id"],
                "query_image_path": query_df.iloc[qi]["image_path"],
                "rank": local_rank,
                "candidate_index": int(pi),
                "candidate_image_id": db_df.iloc[int(pi)]["image_id"],
                "candidate_image_path": db_df.iloc[int(pi)]["image_path"],
                "visual_score_raw": float(visual[int(pi)]),
                "visual_score_norm": float(v_norm[order][local_rank - 1]),
                "predicted_cui_score": float(pred_sim[order][local_rank - 1]),
                "kg_score": float(kg_score[order][local_rank - 1]),
                "final_score": float(ranked_scores[local_rank - 1]),
                "alpha_visual": float(alpha),
                "beta_predicted_cui": float(beta),
                "gamma_kg": float(gamma),
                "query_confidence": float(conf),
                "predicted_query_cuis": ";".join(q_pred_names),
                "candidate_predicted_cuis": ";".join(inv_vocab[i] for i, _ in cand_preds if i < len(inv_vocab)),
                "shared_gt_cuis": ";".join(inv_vocab[i] for i in shared if i < len(inv_vocab)),
                "graph_expanded_cuis": ";".join(expanded_names),
                "gt_jaccard": float(jaccard_sets(q_gt, c_gt)),
            })

    explanations = pd.DataFrame(explanation_rows)
    return all_indices, all_scores, explanations


def evaluate_retrieval(
    query_df: pd.DataFrame,
    db_df: pd.DataFrame,
    top_idx: np.ndarray,
    top_scores: np.ndarray,
    n_labels: int,
    cfg: CFG,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    query_sets = [set(x) for x in query_df["label_indices"].tolist()]
    db_sets = [set(x) for x in db_df["label_indices"].tolist()]
    qY = make_sparse_label_matrix(query_df, n_labels)
    dY = make_sparse_label_matrix(db_df, n_labels)
    d_sizes = np.asarray(dY.sum(axis=1)).ravel().astype(np.float32)
    same_pool = cfg.SAME_SPLIT_SELF_EXCLUDE and list(query_df["image_id"]) == list(db_df["image_id"])

    rows = []
    k_curve = list(range(1, cfg.K_CURVE_MAX + 1))
    for qi in tqdm(range(len(query_df)), desc="Evaluating metrics"):
        qset = query_sets[qi]
        q_size = len(qset)
        inter = (qY[qi] @ dY.T).toarray().ravel().astype(np.float32)
        union = q_size + d_sizes - inter
        all_jac = np.divide(inter, np.maximum(union, 1.0), out=np.zeros_like(inter), where=union > 0)
        if same_pool:
            all_jac[qi] = 0.0
            inter[qi] = 0.0
        ideal_gains = np.sort(all_jac)[::-1]
        total_relevant = int((inter > 0).sum())

        retrieved = top_idx[qi]
        retrieved_jac = [float(all_jac[int(pi)]) for pi in retrieved]
        binary_rel = [j > 0 for j in retrieved_jac]

        row = {
            "query_index": qi,
            "query_image_id": query_df.iloc[qi]["image_id"],
            "n_query_cuis": int(q_size),
            "total_relevant_in_database": int(total_relevant),
            "top1_final_score": float(top_scores[qi, 0]),
        }
        for k in k_curve:
            kk = min(k, len(retrieved))
            rel_k = binary_rel[:kk]
            jac_k = retrieved_jac[:kk]
            retrieved_cuis = set()
            for pi in retrieved[:kk]:
                retrieved_cuis |= db_sets[int(pi)]


            row[f"CUI@{k}"] = float(any(rel_k))

            row[f"NDCG@{k}"] = ndcg_at_k(jac_k, ideal_gains, kk)
            row[f"MeanJaccard@{k}"] = float(np.mean(jac_k)) if kk > 0 else np.nan

            row[f"CUI-Coverage@{k}"] = len(qset & retrieved_cuis) / max(len(qset), 1)
            row[f"ConceptRecall@{k}"] = row[f"CUI-Coverage@{k}"]
            row[f"Precision@{k}"] = float(np.mean(rel_k)) if kk > 0 else np.nan
            row[f"Recall@{k}"] = float(np.sum(rel_k) / max(total_relevant, 1)) if total_relevant > 0 else np.nan
            row[f"mAP@{k}"] = average_precision_at_k(binary_rel, total_relevant, kk)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(binary_rel, kk)
        rows.append(row)

    per_query = pd.DataFrame(rows)
    metric_rows = []
    metric_names = ["CUI", "NDCG", "MeanJaccard", "CUI-Coverage", "Precision", "Recall", "mAP", "MRR"]
    rng = np.random.default_rng(123)
    for metric in metric_names:
        for k in k_curve:
            col = f"{metric}@{k}"
            vals = per_query[col].dropna().values.astype(np.float64)
            if len(vals) == 0:
                continue
            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            if cfg.BOOTSTRAP_ROUNDS and len(vals) > 1:
                boot = []
                for _ in range(cfg.BOOTSTRAP_ROUNDS):
                    sample = rng.choice(vals, size=len(vals), replace=True)
                    boot.append(np.mean(sample))
                ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
            else:
                ci_low, ci_high = mean, mean
            metric_rows.append({
                "metric": metric,
                "K": k,
                "mean": mean,
                "std": std,
                "ci95_low": float(ci_low),
                "ci95_high": float(ci_high),
                "mean_std": f"{mean:.4f} +/- {std:.4f}",
                "mean_95ci": f"{mean:.4f} [{ci_low:.4f}, {ci_high:.4f}]",
                "n_queries": int(len(vals)),
            })
    summary = pd.DataFrame(metric_rows)
    return per_query, summary


def plot_metric_curves(summary: pd.DataFrame, fig_dir: str) -> None:
    def plot_one(metrics: List[str], name: str, title: str):
        plt.figure(figsize=(7, 4))
        for metric in metrics:
            temp = summary[summary["metric"] == metric].sort_values("K")
            if len(temp) == 0:
                continue
            plt.plot(temp["K"], temp["mean"], marker="o", label=metric)
            plt.fill_between(temp["K"], temp["ci95_low"], temp["ci95_high"], alpha=0.15)
        plt.xlabel("K")
        plt.ylabel("Score")
        plt.title(title)
        plt.grid(True, alpha=0.3)
        plt.legend()
        savefig(fig_dir, name)

    plot_one(["CUI"], "02_cui_at_k_curve.png", "CUI@K curve")
    plot_one(["NDCG"], "03_ndcg_at_k_curve.png", "NDCG@K curve")
    plot_one(["Precision", "Recall"], "04_precision_recall_at_k_curve.png", "Precision@K and Recall@K curve")
    plot_one(["mAP", "MRR"], "05_map_mrr_at_k_curve.png", "mAP@K and MRR@K curve")
    plot_one(["MeanJaccard"], "06_mean_jaccard_at_k_curve.png", "Mean Jaccard@K curve")
    plot_one(["CUI-Coverage"], "07_cui_coverage_at_k_curve.png", "CUI-Coverage@K / Concept Recall@K curve")

    main = summary[summary["K"].isin([5, 10, 50])]
    if len(main):
        pivot = main.pivot(index="metric", columns="K", values="mean")
        pivot = pivot.reindex(["CUI", "NDCG", "Precision", "Recall", "mAP", "MRR", "MeanJaccard", "CUI-Coverage"])
        pivot.plot(kind="bar", figsize=(10, 5))
        plt.title("Main retrieval metrics")
        plt.ylabel("Mean score")
        plt.xticks(rotation=30, ha="right")
        savefig(fig_dir, "08_main_metrics_barplot.png")


def make_embedding_visualization(emb: np.ndarray, df: pd.DataFrame, fig_dir: str, cfg: CFG, seed: int) -> None:
    n = min(cfg.TSNE_MAX_POINTS, len(df))
    if n < 10:
        return
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(df), size=n, replace=False)
    X = emb[idx]
    labels = [str(df.iloc[i]["label_indices"][0]) if len(df.iloc[i]["label_indices"]) else "none" for i in idx]

    cnt = Counter(labels)
    common = {c: j for j, (c, _) in enumerate(cnt.most_common(10))}
    y = np.array([common.get(c, 10) for c in labels])

    if TSNE is not None:
        if PCA is not None and X.shape[1] > 50:
            X0 = PCA(n_components=50, random_state=seed).fit_transform(X)
        else:
            X0 = X
        X2 = TSNE(n_components=2, perplexity=min(30, max(5, n // 50)), init="pca", learning_rate="auto", random_state=seed).fit_transform(X0)
        title = "t-SNE embedding visualization"
        fname = "09_tsne_embedding_visualization.png"
    elif PCA is not None:
        X2 = PCA(n_components=2, random_state=seed).fit_transform(X)
        title = "PCA embedding visualization"
        fname = "09_pca_embedding_visualization.png"
    else:
        return

    plt.figure(figsize=(7, 6))
    plt.scatter(X2[:, 0], X2[:, 1], c=y, s=8, alpha=0.75)
    plt.title(title)
    plt.xlabel("Dim 1")
    plt.ylabel("Dim 2")
    plt.grid(True, alpha=0.2)
    savefig(fig_dir, fname)


def load_thumb(path: str, size: int = 160) -> Image.Image:
    try:
        img = Image.open(path).convert("RGB")
        img = ImageOps.contain(img, (size, size))
        canvas = Image.new("RGB", (size, size), "white")
        canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
        return canvas
    except Exception:
        return Image.new("RGB", (size, size), "white")


def retrieval_grid(
    query_df: pd.DataFrame,
    db_df: pd.DataFrame,
    top_idx: np.ndarray,
    per_query: pd.DataFrame,
    fig_dir: str,
    cfg: CFG,
    name: str,
    query_indices: List[int],
    title: str,
    top_k: int = 5,
) -> None:
    if not query_indices:
        return
    top_k = min(top_k, top_idx.shape[1])
    rows = len(query_indices)
    cols = top_k + 1
    fig, axes = plt.subplots(rows, cols, figsize=(2.1 * cols, 2.4 * rows))
    if rows == 1:
        axes = np.expand_dims(axes, 0)
    for r, qi in enumerate(query_indices):
        q_img = load_thumb(query_df.iloc[qi]["image_path"])
        axes[r, 0].imshow(q_img)
        axes[r, 0].set_title(f"Query\n{query_df.iloc[qi]['image_id'][:14]}", fontsize=8)
        axes[r, 0].axis("off")
        qset = set(query_df.iloc[qi]["label_indices"])
        for c in range(top_k):
            pi = int(top_idx[qi, c])
            img = load_thumb(db_df.iloc[pi]["image_path"])
            axes[r, c + 1].imshow(img)
            cand_set = set(db_df.iloc[pi]["label_indices"])
            jac = jaccard_sets(qset, cand_set)
            axes[r, c + 1].set_title(f"R{c+1} J={jac:.2f}\n{db_df.iloc[pi]['image_id'][:14]}", fontsize=8)
            axes[r, c + 1].axis("off")
    plt.suptitle(title, y=1.01, fontsize=12)
    savefig(fig_dir, name)


def make_example_figures(query_df, db_df, top_idx, per_query, emb, fig_dir, cfg, seed):
    rng = np.random.default_rng(seed)
    if len(query_df) > 0:
        sample_q = rng.choice(len(query_df), size=min(cfg.RANDOM_EXAMPLE_QUERIES, len(query_df)), replace=False).tolist()
        retrieval_grid(query_df, db_df, top_idx, per_query, fig_dir, cfg, "10_top5_retrieval_examples.png", sample_q, "Top-5 retrieval examples", top_k=5)
        retrieval_grid(query_df, db_df, top_idx, per_query, fig_dir, cfg, "11_top10_retrieval_examples.png", sample_q[:min(3, len(sample_q))], "Top-10 retrieval examples", top_k=10)

    fail_col = "Precision@5" if "Precision@5" in per_query.columns else f"Precision@{cfg.K_VALUES[0]}"
    failures = per_query.sort_values([fail_col, "MeanJaccard@5" if "MeanJaccard@5" in per_query.columns else fail_col]).head(cfg.FAILURE_EXAMPLE_QUERIES)
    fail_q = failures["query_index"].astype(int).tolist()
    retrieval_grid(query_df, db_df, top_idx, per_query, fig_dir, cfg, "12_failure_case_retrieval_examples.png", fail_q, "Failure-case retrieval examples", top_k=5)
    make_embedding_visualization(emb, query_df, fig_dir, cfg, seed)


def inspect_medimageinsight_model(classifier) -> None:
    print("\nMedImageInsight object type:", type(classifier))
    print("classifier.model type:", type(classifier.model))
    print("Top-level classifier attributes:", [a for a in dir(classifier) if not a.startswith("_")][:80])
    print("Top-level model children:")
    for name, child in list(classifier.model.named_children())[:30]:
        n_params = sum(p.numel() for p in child.parameters()) if hasattr(child, "parameters") else 0
        print(f"  {name}: {type(child).__name__}, params={n_params:,}")


def find_likely_vision_module(model: nn.Module) -> nn.Module:
    names = [
        "vision_model", "vision", "visual", "image_encoder", "vision_encoder",
        "encoder", "image_model", "model"
    ]
    for name in names:
        if hasattr(model, name):
            obj = getattr(model, name)
            if isinstance(obj, nn.Module):
                return obj
    return model


def set_medimageinsight_trainability(model: nn.Module, cfg: CFG) -> int:
    """Unfreeze MedImageInsight encoder parameters according to cfg."""
    for p in model.parameters():
        p.requires_grad = False

    if not cfg.TRAIN_MEDIMAGEINSIGHT_ENCODER:
        return 0

    if cfg.FINETUNE_MODE.lower() == "all":
        for p in model.parameters():
            p.requires_grad = True
    else:
        vision = find_likely_vision_module(model)
        children = list(vision.named_children())

        for name, p in vision.named_parameters():
            lname = name.lower()
            if any(tok in lname for tok in ["norm", "ln", "layernorm", "proj", "head"]):
                p.requires_grad = True
        if children:
            for child_name, child in children[-int(cfg.UNFREEZE_LAST_N_CHILDREN):]:
                for p in child.parameters():
                    p.requires_grad = True
        else:

            named = list(vision.named_parameters())
            cut = max(1, int(0.20 * len(named)))
            for _, p in named[-cut:]:
                p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"MedImageInsight trainable parameters: {trainable:,}/{total:,} ({100*trainable/max(total,1):.2f}%)")
    if trainable == 0:
        raise RuntimeError("No MedImageInsight encoder parameters are trainable. Set cfg.FINETUNE_MODE='all' or inspect the model.")
    return trainable


def get_train_transforms(cfg: CFG):
    try:
        from torchvision import transforms
    except Exception as e:
        raise ImportError("torchvision is required for image tensor training. Install torchvision.") from e
    return transforms.Compose([
        transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])


def get_eval_transforms(cfg: CFG):
    try:
        from torchvision import transforms
    except Exception as e:
        raise ImportError("torchvision is required for image tensor training. Install torchvision.") from e
    return transforms.Compose([
        transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])


class ImageCUIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, n_labels: int, transform=None):
        self.df = df.reset_index(drop=True)
        self.n_labels = n_labels
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["image_path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), "white")
        if self.transform is not None:
            img = self.transform(img)
        y = torch.zeros(self.n_labels, dtype=torch.float32)
        y[row["label_indices"]] = 1.0
        return img, y, str(row["image_id"])


def collate_image_cui(batch):
    imgs, ys, ids = zip(*batch)
    return torch.stack(list(imgs), dim=0), torch.stack(list(ys), dim=0), list(ids)


def _extract_tensor_from_output(out: Any) -> torch.Tensor:
    """Converts common model outputs into a 2D [B,D] embedding tensor."""
    if isinstance(out, torch.Tensor):
        x = out
    elif isinstance(out, dict):
        keys = [
            "image_embeddings", "image_embedding", "image_embeds", "image_features",
            "embeddings", "features", "pooler_output", "pooled_output", "last_hidden_state"
        ]
        x = None
        for k in keys:
            if k in out and isinstance(out[k], torch.Tensor):
                x = out[k]
                break
        if x is None:
            for v in out.values():
                if isinstance(v, torch.Tensor):
                    x = v
                    break
        if x is None:
            raise RuntimeError(f"Could not find tensor in model dict output keys={list(out.keys())}")
    elif hasattr(out, "image_embeds"):
        x = out.image_embeds
    elif hasattr(out, "image_embeddings"):
        x = out.image_embeddings
    elif hasattr(out, "pooler_output"):
        x = out.pooler_output
    elif hasattr(out, "last_hidden_state"):
        x = out.last_hidden_state
    elif isinstance(out, (tuple, list)):
        tensors = [v for v in out if isinstance(v, torch.Tensor)]
        if not tensors:
            raise RuntimeError("Model output tuple/list has no tensor.")
        x = tensors[0]
    else:
        raise RuntimeError(f"Unsupported model output type: {type(out)}")

    if x.ndim == 4:
        x = x.mean(dim=(2, 3))
    elif x.ndim == 3:

        x = x[:, 0] if x.shape[1] > 1 else x.mean(dim=1)
    elif x.ndim == 1:
        x = x[None, :]
    if x.ndim != 2:
        raise RuntimeError(f"Could not convert model output to [B,D], got shape {tuple(x.shape)}")
    return x.float()


def _try_model_call(model: nn.Module, images: torch.Tensor, name: str):
    if name == "encode_image" and hasattr(model, "encode_image"):
        return model.encode_image(images)
    if name == "get_image_features" and hasattr(model, "get_image_features"):
        return model.get_image_features(images)
    if name == "forward_image" and hasattr(model, "forward_image"):
        return model.forward_image(images)
    if name == "forward_vision" and hasattr(model, "forward_vision"):
        return model.forward_vision(images)
    if name == "vision_model" and hasattr(model, "vision_model"):
        return model.vision_model(images)
    if name == "visual" and hasattr(model, "visual"):
        return model.visual(images)
    if name == "image_encoder" and hasattr(model, "image_encoder"):
        return model.image_encoder(images)
    if name == "pixel_values_kw":
        return model(pixel_values=images)
    if name == "images_kw":
        return model(images=images)
    if name == "image_kw":
        return model(image=images)
    if name == "dict_images":
        return model({"images": images})
    if name == "dict_image":
        return model({"image": images})
    if name == "direct":
        return model(images)
    raise AttributeError(name)


def resolve_vision_forward(model: nn.Module, sample_images: torch.Tensor):
    candidates = [
        "encode_image", "get_image_features", "forward_image", "forward_vision",
        "vision_model", "visual", "image_encoder", "pixel_values_kw", "images_kw",
        "image_kw", "dict_images", "dict_image", "direct"
    ]
    errors = []
    was_training = model.training
    model.train(was_training)
    for name in candidates:
        try:
            out = _try_model_call(model, sample_images, name)
            emb = _extract_tensor_from_output(out)
            if emb.shape[0] == sample_images.shape[0] and emb.shape[1] > 8:
                if not emb.requires_grad and emb.grad_fn is None:
                    errors.append((name, f"embedding shape {tuple(emb.shape)} but output is detached/no_grad, not trainable"))
                    continue
                print(f"Resolved differentiable MedImageInsight image forward path: {name}; embedding dim={emb.shape[1]}")
                def f(images, _name=name):
                    y = _extract_tensor_from_output(_try_model_call(model, images, _name))
                    if torch.is_grad_enabled() and (not y.requires_grad and y.grad_fn is None):
                        raise RuntimeError(f"Forward path {_name} returned detached embeddings; encoder cannot train through it.")
                    return y
                return f, int(emb.shape[1]), name
            errors.append((name, f"bad embedding shape {tuple(emb.shape)}"))
        except Exception as e:
            errors.append((name, repr(e)[:350]))
    print("\nCould not resolve a differentiable tensor forward path for classifier.model.")
    print("Tried paths and errors:")
    for name, err in errors:
        print(f"  {name}: {err}")
    print("\nModel inspection:")
    for n, ch in list(model.named_children())[:50]:
        print(" ", n, type(ch))
    raise RuntimeError(
        "MedImageInsight encoder training requires a differentiable tensor image-forward path. "
        "The installed wrapper appears to expose only encode()/predict() inference. "
        "Use the frozen MedImageInsight pipeline or edit forward_medimageinsight_tensor() for this package version."
    )


class CUIFineTuneHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int, n_labels: int, dropout: float = 0.2):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.projector = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, proj_dim),
        )
        self.classifier = nn.Linear(proj_dim, n_labels)
    def forward(self, raw_emb: torch.Tensor):
        z = self.projector(self.norm(raw_emb.float()))
        z = F.normalize(z, dim=1)
        logits = self.classifier(z)
        return z, logits


def build_kg_affinity_matrix(graph: Dict[str, Any], n_labels: int, cfg: CFG) -> np.ndarray:
    W = np.eye(n_labels, dtype=np.float32)
    for a, neigh in graph.get("adjacency", {}).items():
        a = int(a)
        if not neigh:
            continue
        max_w = max(float(w) for _, w in neigh)
        for b, w in neigh:
            b = int(b)
            val = cfg.KG_EXPAND_DECAY * float(w) / max(max_w, 1e-12)
            if 0 <= a < n_labels and 0 <= b < n_labels:
                W[a, b] = max(W[a, b], val)

    W = np.maximum(W, W.T)
    return W.astype("float32")


def kg_rank_loss(emb: torch.Tensor, labels: torch.Tensor, kg_affinity: torch.Tensor, margin: float = 0.15) -> torch.Tensor:
    """KG-aware ranking loss: high KG/CUI related samples should be closer than unrelated samples."""
    b = emb.size(0)
    if b <= 2:
        return emb.float().new_tensor(0.0)
    emb_f = F.normalize(emb.float(), dim=1)
    labels_f = labels.float()
    sim = emb_f @ emb_f.T

    target = labels_f @ kg_affinity @ labels_f.T
    denom = torch.clamp(labels_f.sum(dim=1, keepdim=True) @ labels_f.sum(dim=1, keepdim=True).T, min=1.0)
    target = target / denom
    eye = torch.eye(b, device=emb.device, dtype=torch.bool)
    pos_mask = (target > 0.0) & (~eye)
    neg_mask = (target <= 0.0) & (~eye)
    losses = []
    for i in range(b):
        if pos_mask[i].sum() == 0 or neg_mask[i].sum() == 0:
            continue

        pos_sim = sim[i][pos_mask[i]].min()
        neg_sim = sim[i][neg_mask[i]].max()
        losses.append(F.relu(float(margin) + neg_sim - pos_sim))
    if not losses:
        return emb_f.new_tensor(0.0)
    return torch.stack(losses).mean()


def train_one_seed_end2end(
    seed: int,
    dfs: Dict[str, pd.DataFrame],
    n_labels: int,
    graph: Dict[str, Any],
    cfg: CFG,
    out_dir: str,
    fig_dir: str,
    device: torch.device,
):
    seed_everything(seed)
    print(f"\n===== Seed {seed}: loading fresh MedImageInsight =====")
    classifier = load_medimageinsight(cfg, device)
    classifier.model.to(device)
    inspect_medimageinsight_model(classifier)
    set_medimageinsight_trainability(classifier.model, cfg)

    train_tf = get_train_transforms(cfg)
    eval_tf = get_eval_transforms(cfg)
    train_ds = ImageCUIDataset(dfs["train"], n_labels, train_tf)
    valid_ds = ImageCUIDataset(dfs["valid"], n_labels, eval_tf)
    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS,
                              pin_memory=(device.type == "cuda"), collate_fn=collate_image_cui, drop_last=True)
    valid_loader = DataLoader(valid_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS,
                              pin_memory=(device.type == "cuda"), collate_fn=collate_image_cui)

    sample_images, _, _ = next(iter(train_loader))
    sample_images = sample_images.to(device, non_blocking=True)
    vision_forward, in_dim, forward_name = resolve_vision_forward(classifier.model, sample_images[:min(2, sample_images.size(0))])

    head = CUIFineTuneHead(in_dim, cfg.PROJECTION_DIM, n_labels, cfg.HEAD_DROPOUT).to(device)
    pos_weight = build_pos_weight(dfs["train"], n_labels, cfg, device)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    kg_affinity = torch.tensor(build_kg_affinity_matrix(graph, n_labels, cfg), dtype=torch.float32, device=device)

    encoder_params = [p for p in classifier.model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": cfg.ENCODER_LR},
        {"params": head.parameters(), "lr": cfg.HEAD_LR},
    ], weight_decay=cfg.WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.USE_AMP and device.type == "cuda"))

    best_valid = float("inf")
    best_path = os.path.join(out_dir, f"best_trainable_medimageinsight_seed_{seed}.pt")
    patience = 0
    history = []

    for epoch in range(1, cfg.EPOCHS + 1):
        classifier.model.train()
        head.train()
        train_losses, train_bce, train_sup, train_cf, train_kg = [], [], [], [], []
        for images, y, _ids in tqdm(train_loader, desc=f"seed {seed} epoch {epoch} train"):
            images = images.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(cfg.USE_AMP and device.type == "cuda")):
                raw_emb = vision_forward(images)
                emb, logits = head(raw_emb)
                loss_bce = bce(logits, y)
                loss_sup = supervised_contrastive_multilabel_loss(emb, y, cfg.SUPCON_TEMP)
                loss_cf = counterfactual_margin_loss(emb, y, cfg.COUNTERFACTUAL_MARGIN)
                loss_kg = kg_rank_loss(emb, y, kg_affinity, cfg.KG_RANK_MARGIN)
                loss = (
                    cfg.LAMBDA_BCE * loss_bce
                    + cfg.LAMBDA_RETRIEVAL * loss_sup
                    + cfg.LAMBDA_COUNTERFACTUAL * loss_cf
                    + cfg.LAMBDA_KG_RANK * loss_kg
                )
            scaler.scale(loss).backward()
            if cfg.GRAD_CLIP is not None and cfg.GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(list(classifier.model.parameters()) + list(head.parameters()), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.detach().cpu()))
            train_bce.append(float(loss_bce.detach().cpu()))
            train_sup.append(float(loss_sup.detach().cpu()))
            train_cf.append(float(loss_cf.detach().cpu()))
            train_kg.append(float(loss_kg.detach().cpu()))

        classifier.model.eval()
        head.eval()
        valid_losses, valid_bces, valid_kgs = [], [], []
        with torch.no_grad():
            for images, y, _ids in tqdm(valid_loader, desc=f"seed {seed} epoch {epoch} valid"):
                images = images.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                raw_emb = vision_forward(images)
                emb, logits = head(raw_emb)
                loss_bce = bce(logits, y)
                loss_kg = kg_rank_loss(emb, y, kg_affinity, cfg.KG_RANK_MARGIN)
                loss = cfg.LAMBDA_BCE * loss_bce + cfg.LAMBDA_KG_RANK * loss_kg
                valid_losses.append(float(loss.detach().cpu()))
                valid_bces.append(float(loss_bce.detach().cpu()))
                valid_kgs.append(float(loss_kg.detach().cpu()))

        row = {
            "seed": seed,
            "epoch": epoch,
            "forward_path": forward_name,
            "train_loss": float(np.mean(train_losses)),
            "train_bce": float(np.mean(train_bce)),
            "train_supcon": float(np.mean(train_sup)),
            "train_counterfactual": float(np.mean(train_cf)),
            "train_kg_rank": float(np.mean(train_kg)),
            "valid_loss": float(np.mean(valid_losses)),
            "valid_bce": float(np.mean(valid_bces)),
            "valid_kg_rank": float(np.mean(valid_kgs)),
        }
        history.append(row)
        print(row)

        if row["valid_loss"] < best_valid:
            best_valid = row["valid_loss"]
            torch.save({
                "medimageinsight_state_dict": classifier.model.state_dict(),
                "head_state_dict": head.state_dict(),
                "in_dim": in_dim,
                "forward_path": forward_name,
                "cfg": asdict(cfg),
            }, best_path)
            patience = 0
        else:
            patience += 1
            if patience >= cfg.EARLY_STOP_PATIENCE:
                print("Early stopping.")
                break

    ckpt = torch.load(best_path, map_location=device)
    classifier.model.load_state_dict(ckpt["medimageinsight_state_dict"])
    head.load_state_dict(ckpt["head_state_dict"])
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(out_dir, f"training_history_seed_{seed}.csv"), index=False)

    plt.figure(figsize=(7, 4))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train total")
    plt.plot(hist_df["epoch"], hist_df["valid_loss"], marker="o", label="valid total")
    plt.plot(hist_df["epoch"], hist_df["train_kg_rank"], marker="o", label="train KG-rank")
    plt.plot(hist_df["epoch"], hist_df["valid_kg_rank"], marker="o", label="valid KG-rank")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training/validation loss curve seed {seed}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    savefig(fig_dir, f"01_training_validation_loss_seed_{seed}.png")
    return classifier, head, vision_forward, hist_df, best_path


@torch.no_grad()
def infer_end2end(classifier, head, vision_forward, df: pd.DataFrame, n_labels: int, cfg: CFG, device: torch.device):
    ds = ImageCUIDataset(df, n_labels, get_eval_transforms(cfg))
    loader = DataLoader(ds, batch_size=max(1, cfg.BATCH_SIZE * 2), shuffle=False, num_workers=cfg.NUM_WORKERS,
                        pin_memory=(device.type == "cuda"), collate_fn=collate_image_cui)
    classifier.model.eval()
    head.eval()
    embs, probs, logits_all, ids = [], [], [], []
    for images, y, batch_ids in tqdm(loader, desc="End-to-end inference"):
        images = images.to(device, non_blocking=True)
        raw_emb = vision_forward(images)
        z, logits = head(raw_emb)
        prob = torch.sigmoid(logits)
        embs.append(z.detach().cpu().numpy().astype("float32"))
        logits_all.append(logits.detach().cpu().numpy().astype("float32"))
        probs.append(prob.detach().cpu().numpy().astype("float32"))
        ids.extend(batch_ids)
    return np.concatenate(embs), np.concatenate(probs), np.concatenate(logits_all), ids


def run_one_seed(
    seed: int,
    dfs: Dict[str, pd.DataFrame],
    vocab: Dict[str, int],
    inv_vocab: List[str],
    graph: Dict[str, Any],
    cfg: CFG,
    out_dir: str,
    device: torch.device,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    seed_dir = ensure_dir(os.path.join(out_dir, f"seed_{seed}"))
    fig_dir = ensure_dir(os.path.join(seed_dir, "figures"))
    n_labels = len(inv_vocab)

    classifier, head, vision_forward, hist_df, best_path = train_one_seed_end2end(
        seed, dfs, n_labels, graph, cfg, seed_dir, fig_dir, device
    )

    emb_map, prob_map, logit_map = {}, {}, {}
    for split in ["train", "valid", "test"]:
        emb, prob, logit, ids = infer_end2end(classifier, head, vision_forward, dfs[split], n_labels, cfg, device)
        emb_map[split] = emb
        prob_map[split] = prob
        logit_map[split] = logit
        np.save(os.path.join(seed_dir, f"{split}_trainable_medimageinsight_embeddings.npy"), emb)
        np.save(os.path.join(seed_dir, f"{split}_predicted_cui_probs.npy"), prob)
        pd.DataFrame({"image_id": ids}).to_csv(os.path.join(seed_dir, f"{split}_embedding_ids.csv"), index=False)

    query_split = cfg.QUERY_SPLIT
    db_split = cfg.DATABASE_SPLIT
    query_df, db_df = dfs[query_split], dfs[db_split]
    top_idx, top_scores, explanations = retrieve_and_explain(
        query_df=query_df,
        db_df=db_df,
        query_emb=emb_map[query_split],
        db_emb=emb_map[db_split],
        query_probs=prob_map[query_split],
        db_probs=prob_map[db_split],
        graph=graph,
        inv_vocab=inv_vocab,
        cfg=cfg,
    )
    np.save(os.path.join(seed_dir, "top_indices.npy"), top_idx)
    np.save(os.path.join(seed_dir, "top_final_scores.npy"), top_scores)
    explanations["seed"] = seed
    explanations.to_csv(os.path.join(seed_dir, "retrieval_explanations.csv"), index=False)

    per_query, summary = evaluate_retrieval(query_df, db_df, top_idx, top_scores, n_labels, cfg)
    per_query["seed"] = seed
    summary["seed"] = seed
    per_query.to_csv(os.path.join(seed_dir, "query_level_metrics.csv"), index=False)
    summary.to_csv(os.path.join(seed_dir, "summary_metrics_by_k.csv"), index=False)

    main_summary = summary[summary["K"].isin(cfg.K_VALUES)].copy()
    main_summary.to_csv(os.path.join(seed_dir, "summary_metrics_main_k.csv"), index=False)
    print("Main metrics for seed", seed)
    display(main_summary.pivot(index="metric", columns="K", values="mean_std"))

    plot_metric_curves(summary, fig_dir)
    make_example_figures(query_df, db_df, top_idx, per_query, emb_map[query_split], fig_dir, cfg, seed)


    del classifier, head
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return per_query, summary, explanations


def plot_seed_level_curves(seed_level: pd.DataFrame, fig_dir: str) -> None:
    def plot_one(metrics: List[str], name: str, title: str):
        plt.figure(figsize=(7, 4))
        for metric in metrics:
            temp = seed_level[seed_level["metric"] == metric].sort_values("K")
            if len(temp) == 0:
                continue
            plt.plot(temp["K"], temp["seed_mean"], marker="o", label=metric)
            plt.fill_between(temp["K"], temp["seed_mean"] - temp["seed_std"], temp["seed_mean"] + temp["seed_std"], alpha=0.15)
        plt.xlabel("K")
        plt.ylabel("Score")
        plt.title(title + " (seed-level mean ± std)")
        plt.grid(True, alpha=0.3)
        plt.legend()
        savefig(fig_dir, name)
    plot_one(["CUI"], "FINAL_02_cui_at_k_curve_seed_level.png", "CUI@K curve")
    plot_one(["NDCG"], "FINAL_03_ndcg_at_k_curve_seed_level.png", "NDCG@K curve")
    plot_one(["Precision", "Recall"], "FINAL_04_precision_recall_at_k_curve_seed_level.png", "Precision@K and Recall@K curve")
    plot_one(["mAP", "MRR"], "FINAL_05_map_mrr_at_k_curve_seed_level.png", "mAP@K and MRR@K curve")
    plot_one(["MeanJaccard"], "FINAL_06_mean_jaccard_at_k_curve_seed_level.png", "Mean Jaccard@K curve")
    plot_one(["CUI-Coverage"], "FINAL_07_cui_coverage_at_k_curve_seed_level.png", "CUI-Coverage@K / Concept Recall@K curve")


def aggregate_seed_results(all_per_query: List[pd.DataFrame], all_summary: List[pd.DataFrame], all_explanations: List[pd.DataFrame], out_dir: str, cfg: CFG):
    per_query = pd.concat(all_per_query, ignore_index=True)
    summary = pd.concat(all_summary, ignore_index=True)
    explanations = pd.concat(all_explanations, ignore_index=True)
    per_query.to_csv(os.path.join(out_dir, "query_level_metrics_all_seeds.csv"), index=False)
    summary.to_csv(os.path.join(out_dir, "summary_metrics_by_k_all_seeds.csv"), index=False)
    explanations.to_csv(os.path.join(out_dir, "retrieval_explanations_all_seeds.csv"), index=False)


    seed_rows = []
    metric_names = ["CUI", "NDCG", "MeanJaccard", "CUI-Coverage", "Precision", "Recall", "mAP", "MRR"]
    for metric in metric_names:
        for k in range(1, cfg.K_CURVE_MAX + 1):
            col = f"{metric}@{k}"
            if col not in per_query.columns:
                continue
            per_seed = per_query.groupby("seed")[col].mean().dropna()
            if len(per_seed) == 0:
                continue
            mean = float(per_seed.mean())
            std = float(per_seed.std(ddof=1)) if len(per_seed) > 1 else 0.0
            seed_rows.append({
                "metric": metric,
                "K": k,
                "seed_mean": mean,
                "seed_std": std,
                "mean_std": f"{mean:.4f} +/- {std:.4f}",
                "n_seeds": int(len(per_seed)),
                "seed_values": json.dumps({str(s): float(v) for s, v in per_seed.items()}),
            })
    seed_level = pd.DataFrame(seed_rows)
    seed_level.to_csv(os.path.join(out_dir, "FINAL_seed_level_metrics_all_k.csv"), index=False)
    main = seed_level[seed_level["K"].isin(cfg.K_VALUES)].copy()
    main.to_csv(os.path.join(out_dir, "FINAL_main_metrics_seed_mean_std.csv"), index=False)
    print("Final main metrics: seed-level mean +/- std")
    display(main.pivot(index="metric", columns="K", values="mean_std"))
    plot_seed_level_curves(seed_level, ensure_dir(os.path.join(out_dir, "figures")))
    return main


def main(cfg: CFG):
    out_dir = ensure_dir(cfg.OUT_DIR)
    ensure_dir(os.path.join(out_dir, "figures"))
    ensure_dir(cfg.MODEL_CACHE_DIR)
    save_json(asdict(cfg), os.path.join(out_dir, "config.json"))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    if device.type != "cuda":
        print("WARNING: Training MedImageInsight encoder on CPU will be extremely slow.")

    print("Loading ROCOv2 split dataset...")
    dfs = load_roco_splits(cfg)
    vocab, inv_vocab = build_cui_vocab(dfs["train"], cfg)
    for split in ["train", "valid", "test"]:
        dfs[split] = add_label_indices(dfs[split], vocab)
        print(split, dfs[split].shape)
    n_labels = len(inv_vocab)
    save_json(inv_vocab, os.path.join(out_dir, "cui_vocab_train_only.json"))

    print("Building train-only CUI graph/hypergraph...")
    graph = build_train_hypergraph(dfs["train"], n_labels, cfg)
    save_json({"cui_freq": graph["cui_freq"]}, os.path.join(out_dir, "train_only_cui_graph_stats.json"))

    all_per_query, all_summary, all_explanations = [], [], []
    for seed in cfg.SEEDS:
        per_query, summary, explanations = run_one_seed(seed, dfs, vocab, inv_vocab, graph, cfg, out_dir, device)
        all_per_query.append(per_query)
        all_summary.append(summary)
        all_explanations.append(explanations)

    final = aggregate_seed_results(all_per_query, all_summary, all_explanations, out_dir, cfg)

    if cfg.SAVE_ZIP:
        zip_path = shutil.make_archive(out_dir, "zip", out_dir)
        print("Saved zip:", zip_path)
    print("Done. Output directory:", out_dir)
    return final


if __name__ == "__main__":
    final_summary = main(cfg)
